# Project 2 — Kaggle-Style Prediction System

## Overview
This notebook trains **multiple ML models** on the Titanic dataset and compares them systematically — the same workflow used in Kaggle competitions.

**Pipeline:**
```
Data → sklearn Pipeline → Cross-Validation → Hyperparameter Tuning → Leaderboard
```

## 1. Theory — Ensemble Methods

### Bias-Variance Tradeoff
The expected test error decomposes as:
$$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Noise}$$

- **High bias** (underfitting): model is too simple, misses patterns
- **High variance** (overfitting): model memorises training data, fails on new data

### Random Forest — Bagging
Random Forest grows $T$ decision trees, each trained on a **bootstrap** sample of the data and a random subset of features. Final prediction is the majority vote:
$$\hat{y} = \text{mode}\left(\hat{y}_1, \hat{y}_2, \ldots, \hat{y}_T\right)$$

Bagging **reduces variance** — averaging many high-variance, low-bias trees.

### Gradient Boosting
Gradient Boosting builds trees **sequentially**. Each tree corrects the residuals of the previous ensemble:
$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$
where $\eta$ is the learning rate and $h_m$ fits the negative gradient (residuals). Boosting **reduces bias**.

### Cross-Validation
K-fold CV splits data into $k$ folds, trains on $k-1$ and validates on 1, rotating $k$ times. The CV estimate is:
$$\text{CV score} = \frac{1}{k} \sum_{i=1}^{k} \text{score}_i$$

This gives a more reliable estimate than a single train/validation split.

### sklearn Pipeline
A `Pipeline` chains preprocessing and modelling steps so that the same transformations are applied consistently during both training and prediction, and CV can be done without leakage.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, learning_curve
)
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Imports done.')

## 2. Data Loading & Feature Selection

In [ ]:
df = sns.load_dataset('titanic')

# Add derived features before setting up the pipeline
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone']    = (df['family_size'] == 1).astype(int)

# Define feature groups — important for ColumnTransformer
NUMERIC_FEATURES = ['age', 'fare', 'pclass', 'family_size', 'is_alone']
CATEGORICAL_FEATURES = ['sex', 'embarked']
TARGET = 'survived'

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 3. sklearn Pipeline Construction
We use `ColumnTransformer` to apply different transformations to numeric and categorical columns, then chain with a classifier.

In [ ]:
# Preprocessing for numeric features: impute missing → scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical features: impute missing → one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine both preprocessors
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,   NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES)
])

print('Preprocessor built.')
print(preprocessor)

## 4. Training Multiple Models
We wrap each classifier in the same preprocessing pipeline, then run 5-fold CV.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []

for name, clf in models.items():
    # Build full pipeline
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])

    # 5-fold cross-validation on training data
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

    # Fit on full training set and evaluate on test set
    pipe.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, pipe.predict(X_test))

    results.append({
        'Model': name,
        'CV Mean Acc': cv_scores.mean(),
        'CV Std':      cv_scores.std(),
        'Test Acc':    test_acc,
        'Pipeline':    pipe  # keep for later use
    })
    print(f'{name:<25} CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  Test: {test_acc:.4f}')

## 5. Leaderboard

In [ ]:
leaderboard = pd.DataFrame([
    {
        'Model':       r['Model'],
        'CV Accuracy': f"{r['CV Mean Acc']:.4f} ± {r['CV Std']:.4f}",
        'Test Acc':    f"{r['Test Acc']:.4f}",
    }
    for r in results
]).sort_values('Test Acc', ascending=False).reset_index(drop=True)

leaderboard.index = leaderboard.index + 1  # rank starts at 1
leaderboard.index.name = 'Rank'
print('=== LEADERBOARD ===')
print(leaderboard.to_string())

## 6. Hyperparameter Tuning (Best Model)
We tune the best-performing model (typically Random Forest or Gradient Boosting) using `GridSearchCV`.

In [ ]:
# Select best pipeline by CV mean accuracy
best_result = max(results, key=lambda r: r['CV Mean Acc'])
best_name   = best_result['Model']
print(f'Best model for tuning: {best_name}')

# Define hyperparameter grid based on model type
if 'Random Forest' in best_name:
    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth':    [None, 5, 10],
        'classifier__min_samples_split': [2, 5],
    }
elif 'Gradient' in best_name:
    param_grid = {
        'classifier__n_estimators':  [100, 200],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__max_depth':     [3, 5],
    }
else:
    param_grid = {
        'classifier__C': [0.1, 1.0, 10.0],
    }

# Rebuild the pipeline for GridSearch
clf_map = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
}
base_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf_map[best_name])])

grid_search = GridSearchCV(
    base_pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\nBest CV accuracy: {grid_search.best_score_:.4f}')
print(f'Best params: {grid_search.best_params_}')

tuned_test_acc = accuracy_score(y_test, grid_search.best_estimator_.predict(X_test))
print(f'Tuned model test accuracy: {tuned_test_acc:.4f}')

## 7. Feature Importance

In [ ]:
# Get feature names after preprocessing
best_pipe = grid_search.best_estimator_
clf_step  = best_pipe.named_steps['classifier']
pre_step  = best_pipe.named_steps['preprocessor']

# Only tree-based models have feature_importances_
if hasattr(clf_step, 'feature_importances_'):
    cat_feature_names = pre_step.named_transformers_['cat']['onehot'].get_feature_names_out(CATEGORICAL_FEATURES)
    all_feature_names = NUMERIC_FEATURES + list(cat_feature_names)

    importance_df = pd.DataFrame({
        'feature':    all_feature_names,
        'importance': clf_step.feature_importances_
    }).sort_values('importance', ascending=True)

    plt.figure(figsize=(8, 6))
    plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
    plt.title(f'Feature Importance — {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Logistic Regression selected — see coefficients in Project 1.')

## 8. Learning Curves
Learning curves show how model performance changes with training set size — useful to diagnose bias vs variance.

In [ ]:
# Use the best tuned model for learning curve
train_sizes, train_scores, val_scores = learning_curve(
    grid_search.best_estimator_,
    X_train, y_train,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
plt.fill_between(train_sizes, val_mean   - val_std,   val_mean   + val_std,   alpha=0.2, color='orange')
plt.plot(train_sizes, train_mean, 'o-', color='blue',   label='Training score')
plt.plot(train_sizes, val_mean,   'o-', color='orange', label='Cross-val score')
plt.xlabel('Training examples')
plt.ylabel('Accuracy')
plt.title(f'Learning Curves — {best_name} (tuned)')
plt.legend()
plt.tight_layout()
plt.show()
print('If the gap between training and CV score is large → overfitting (high variance).')

## Summary

| Concept | Tool used |
|---|---|
| Preprocessing pipeline | `sklearn Pipeline` + `ColumnTransformer` |
| Baseline comparison | 5-fold CV on 3 models |
| Hyperparameter search | `GridSearchCV` |
| Feature importance | Tree model `.feature_importances_` |
| Bias/variance diagnosis | Learning curves |

**Key insight**: Gradient Boosting and Random Forest usually outperform Logistic Regression on tabular data. However, LR is more interpretable and faster to train — important in production settings.